# Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.8-27b")
response = model.invoke("Why do parrots talk?")
response


AIMessage(content='The short answer is: **Parrots don’t actually “talk” in the human sense.** They are mimicking sounds they hear, including human speech, but they generally do not understand the semantic meaning of the words.\n\nHere’s a detailed breakdown of why and how parrots learn to vocalize human speech:\n\n### 1. Vocal Mimicry, Not Language\n- Parrots have a specialized organ in their throat called the **syrinx**, which allows them to produce a wide range of sounds.\n- They learn to reproduce these sounds through **auditory learning** and **vocal practice**, much like a child learning to speak.\n- However, most parrots do not associate words with their meanings. For example, a parrot might say “hello” without understanding that it’s a greeting. It’s learned the sound pattern, not the concept.\n\n### 2. Social Bonding\n- Parrots are highly **social animals** in the wild. In their natural habitats, they use complex vocalizations to stay in contact with their flock, signal danger,

In [7]:
from langchain_core.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])


In [8]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


content='' additional_kwargs={'tool_calls': [{'id': 'fx6eh0jb2', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 278, 'total_tokens': 304, 'completion_time': 0.067934547, 'completion_tokens_details': None, 'prompt_time': 0.018806261, 'prompt_tokens_details': None, 'queue_time': 0.057084226, 'total_time': 0.086740808}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1e81a-ccf3-7e53-8b54-f6eb63a3502c-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fx6eh0jb2', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 278, 'output_tokens': 26, 'total_tokens': 304}
Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loops


In [9]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.content)
# "The current weather in Boston is 72°F and sunny."


It's sunny in Boston! ☀️


In [ ]:
messages